# T-MVAL Sampling and Verification

This notebook samples DNA sequences from a fine-tuned diffusion checkpoint and validates them with the same families of metrics used in the project verification workflow:

- activity score with Enformer
- ATAC positive rate
- motif distribution Spearman correlation
- Shannon entropy diversity
- 3-mer Pearson correlation


In [1]:
from __future__ import annotations

import collections
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from scipy.stats import pearsonr


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Diffusion").exists() and (candidate / "checkpoints").exists():
            return candidate
    raise FileNotFoundError("Could not find the T-MVAL project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("HOME", "/tmp")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/.cache")
os.environ.setdefault("WANDB_DIR", "/tmp")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

torch.set_float32_matmul_precision("medium")

print(f"Project root: {PROJECT_ROOT}")
print(f"Torch device available: {'cuda:0' if torch.cuda.is_available() else 'cpu'}")


Project root: /root/T-MVAL
Torch device available: cuda:0


In [2]:
DIFFUSION_CHECKPOINT = PROJECT_ROOT / "best.pt"
ACTIVITY_CHECKPOINT = PROJECT_ROOT / "checkpoints" / "oracle_eval.ckpt"
ATAC_CHECKPOINT = PROJECT_ROOT / "checkpoints" / "epoch=3-step=3204.ckpt"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
SAMPLE_BATCH_SIZE = 640
SEQUENCE_LENGTH = 200

OUTPUT_DIR = PROJECT_ROOT / "tutorials" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"{DIFFUSION_CHECKPOINT.stem}_samples.txt"

REFERENCE_SEQUENCE_PATH = PROJECT_ROOT / "top_0_1pct_reference_sequences.txt"

print(f"Diffusion checkpoint: {DIFFUSION_CHECKPOINT}")
print(f"Device: {DEVICE}")
print(f"Samples per run: {SAMPLE_BATCH_SIZE}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Activity checkpoint: {ACTIVITY_CHECKPOINT}")
print(f"ATAC checkpoint: {ATAC_CHECKPOINT}")
print(f"Reference sequence path: {REFERENCE_SEQUENCE_PATH}")


Diffusion checkpoint: /root/T-MVAL/best.pt
Device: cuda:0
Samples per run: 640
Output path: /root/T-MVAL/tutorials/outputs/best_samples.txt
Activity checkpoint: /root/T-MVAL/checkpoints/oracle_eval.ckpt
ATAC checkpoint: /root/T-MVAL/checkpoints/epoch=3-step=3204.ckpt
Reference sequence path: /root/T-MVAL/top_0_1pct_reference_sequences.txt


In [3]:
from Diffusion.diffusion import Diffusion
from Diffusion.unet import UNet

DNA_MAPPING = np.array(["A", "C", "G", "T"])


def load_diffusion_checkpoint(checkpoint_path: Path, device: str = DEVICE) -> Diffusion:
    checkpoint_state = torch.load(checkpoint_path, map_location=device)
    model = UNet(dim=200, channels=1, dim_mults=[1, 2, 4], resnet_block_groups=4)
    diffusion = Diffusion(model, timesteps=50, beta_start=0.0001, beta_end=0.2)

    if isinstance(checkpoint_state, dict) and "model" in checkpoint_state:
        diffusion.load_state_dict(checkpoint_state["model"])
    else:
        diffusion.load_state_dict(checkpoint_state)

    diffusion.to(device)
    diffusion.eval()
    return diffusion


def logits_to_sequences(logits: torch.Tensor) -> list[str]:
    token_ids = torch.argmax(logits, dim=1).cpu().numpy()
    return ["".join(DNA_MAPPING[row]) for row in token_ids]


def sample_sequences_from_checkpoint(
    checkpoint_path: Path,
    batch_size: int = SAMPLE_BATCH_SIZE,
    device: str = DEVICE,
    output_path: Path | None = None,
):
    diffusion = load_diffusion_checkpoint(checkpoint_path, device=device)

    with torch.no_grad():
        final_logits = diffusion.p_sample_loop(
            classes=None,
            image_size=(batch_size, 1, 4, SEQUENCE_LENGTH),
            cond_weight=1,
        )[-1]

    final_logits = torch.from_numpy(np.array(final_logits)).squeeze(1)
    sequences = logits_to_sequences(final_logits)

    if output_path is not None:
        output_path.write_text("\n".join(sequences) + "\n")

    return sequences, final_logits


In [4]:
generated_sequences, sampled_logits = sample_sequences_from_checkpoint(
    DIFFUSION_CHECKPOINT,
    batch_size=SAMPLE_BATCH_SIZE,
    device=DEVICE,
    output_path=OUTPUT_PATH,
)

print(f"Generated {len(generated_sequences)} sequences.")
print(f"Saved to: {OUTPUT_PATH}")


Generated 640 sequences.
Saved to: /root/T-MVAL/tutorials/outputs/best_samples.txt


In [5]:
def read_sequences(path: Path) -> list[str]:
    return [line.strip().upper() for line in Path(path).read_text().splitlines() if line.strip()]


def sequences_to_one_hot_torch(
    sequences: list[str],
    *,
    channels_first: bool,
    device: str,
) -> torch.Tensor:
    alphabet = {"A": 0, "C": 1, "G": 2, "T": 3}
    tokenized = torch.tensor(
        [[alphabet[base] for base in sequence] for sequence in sequences],
        dtype=torch.long,
        device=device,
    )
    one_hot = F.one_hot(tokenized, num_classes=4).float()
    if channels_first:
        return one_hot.permute(0, 2, 1).contiguous()
    return one_hot


def load_activity_model(checkpoint_path: Path, device: str = DEVICE):
    from enformer_regressor import EnformerModel

    model = EnformerModel.load_from_checkpoint(str(checkpoint_path), map_location=device)
    model = model.to(device)
    model.eval()
    return model


def load_atac_model(checkpoint_path: Path, device: str = DEVICE):
    from grelu.lightning import LightningModel

    model = LightningModel.load_from_checkpoint(str(checkpoint_path), map_location=device, weights_only=False)
    model = model.to(device)
    model.eval()
    return model


def count_kmers(sequences: list[str], k: int = 3) -> collections.Counter:
    counts = collections.Counter()
    for sequence in sequences:
        for index in range(len(sequence) - k + 1):
            counts[sequence[index:index + k]] += 1
    return counts


def kmer_pearson_correlation(
    reference_sequences: list[str],
    generated_sequences: list[str],
    k: int = 3,
) -> float:
    reference_counts = count_kmers(reference_sequences, k=k)
    generated_counts = count_kmers(generated_sequences, k=k)
    kmer_set = sorted(set(reference_counts) | set(generated_counts))

    if not kmer_set:
        return float("nan")

    counts = np.zeros((len(kmer_set), 2), dtype=np.float64)
    for idx, kmer in enumerate(kmer_set):
        if kmer in reference_counts:
            counts[idx, 1] = reference_counts[kmer] * len(generated_sequences) / len(reference_sequences)
        if kmer in generated_counts:
            counts[idx, 0] = generated_counts[kmer]

    return float(pearsonr(counts[:, 0], counts[:, 1])[0])


def calculate_diversity_shannon_entropy(sequences: list[str]) -> float:
    if not sequences:
        return 0.0

    seq_length = len(sequences[0])
    if not all(len(sequence) == seq_length for sequence in sequences):
        raise ValueError("All sequences must have the same length.")

    sequence_array = np.array([list(sequence) for sequence in sequences])
    total_entropy = 0.0

    for index in range(seq_length):
        counts = collections.Counter(sequence_array[:, index])
        position_entropy = 0.0
        for base in ("A", "C", "G", "T"):
            count = counts.get(base, 0)
            if count > 0:
                probability = count / len(sequences)
                position_entropy -= probability * np.log2(probability)
        total_entropy += position_entropy

    return float(total_entropy / seq_length)


def motif_spearman_correlation(
    reference_sequences: list[str],
    generated_sequences: list[str],
):
    try:
        from grelu.interpret.motifs import scan_sequences
        from grelu.io.motifs import get_jaspar

        motif_db = get_jaspar(species="human")
        motif_count_gen = scan_sequences(generated_sequences, motif_db)
        motif_count_ref = scan_sequences(reference_sequences, motif_db)

        motif_sum_gen = motif_count_gen["motif"].value_counts()
        motif_sum_ref = motif_count_ref["motif"].value_counts()

        motif_summary = pd.concat([motif_sum_gen, motif_sum_ref], axis=1).fillna(0)
        motif_summary.columns = ["generated", "reference"]

        correlation = motif_summary.corr(method="spearman").iloc[0, 1]
        return float(correlation), motif_summary
    except Exception as exc:
        warnings.warn(f"Motif validation skipped: {exc}")
        return float("nan"), pd.DataFrame(columns=["generated", "reference"])


In [6]:
def evaluate_sequences(
    generated_sequences: list[str],
    *,
    activity_checkpoint: Path = ACTIVITY_CHECKPOINT,
    atac_checkpoint: Path = ATAC_CHECKPOINT,
    reference_sequence_path: Path = REFERENCE_SEQUENCE_PATH,
    device: str = DEVICE,
):
    reference_sequences = read_sequences(reference_sequence_path)

    activity_model = load_activity_model(activity_checkpoint, device=device)
    atac_model = load_atac_model(atac_checkpoint, device=device)

    activity_inputs = sequences_to_one_hot_torch(
        generated_sequences,
        channels_first=False,
        device=device,
    )
    atac_inputs = sequences_to_one_hot_torch(
        generated_sequences,
        channels_first=True,
        device=device,
    )

    with torch.no_grad():
        activity_scores = activity_model(activity_inputs).detach().cpu()
        atac_scores = atac_model(atac_inputs).detach().cpu()

    motif_corr, motif_summary = motif_spearman_correlation(reference_sequences, generated_sequences)

    metrics = pd.Series(
        {
            "num_sequences": len(generated_sequences),
            "activity_mean": float(activity_scores.mean()),
            "activity_median": float(activity_scores.median()),
            "atac_positive_rate": float((atac_scores[:, 1] > 0.5).float().mean()),
            "shannon_entropy": calculate_diversity_shannon_entropy(generated_sequences),
            "kmer_pearson": kmer_pearson_correlation(reference_sequences, generated_sequences, k=3),
            "motif_spearman": motif_corr,
        },
        name="value",
    )

    artifacts = {
        "activity_scores": activity_scores.numpy(),
        "atac_scores": atac_scores.numpy(),
        "motif_summary": motif_summary,
        "reference_sequences": reference_sequences,
    }
    return metrics, artifacts


In [7]:
metrics, artifacts = evaluate_sequences(generated_sequences)

display(metrics.to_frame())

print("Validation complete.")
print(f"Sample file: {OUTPUT_PATH}")
print(f"Reference sequence file: {REFERENCE_SEQUENCE_PATH}")

/root/miniconda3/envs/fina_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/fina_env/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/root/miniconda3/envs/fina_env/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribu

Loading local artifact from: /root/human_state_dict_local/human.h5


/root/miniconda3/envs/fina_env/lib/python3.10/site-packages/memelite/fimo.py:406: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  hits = pandas.concat(hits)
/root/miniconda3/envs/fina_env/lib/python3.10/site-packages/memelite/fimo.py:406: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  hits = pandas.concat(hits)


,value
num_sequences,640.000000
activity_mean,7.446642
activity_median,7.481866
atac_positive_rate,0.946875
shannon_entropy,1.822244
kmer_pearson,0.972242
motif_spearman,0.937018


Validation complete.
Sample file: /root/T-MVAL/tutorials/outputs/best_samples.txt
Reference sequence file: /root/T-MVAL/top_0_1pct_reference_sequences.txt


In [20]:
def build_signal_change_summary(replacement_sequence: str, api_key: str):
    from alphagenome.data import genome
    from alphagenome.models import dna_client

    wt_sequence_str = (
        "GTAGGCGCCCCATAAGTATTTGCGGAATGAGCAAATACAGATGGGGACAGCTGAGTCAGTAGGGTGAGCAGCT"
        "GGATGGGCCAGTAGTGGAGGGAGGAAACGGCAGCCTCTGCCTGGGCCTTGCCCTGGGTAAACAGACCTCTCTCT"
        "CCCCAGACAGCCACTGTTTGACGACTGTCTGCCATGGGCCTGGCTGAGCCTGG"
    )

    if len(replacement_sequence) != len(wt_sequence_str):
        raise ValueError(
            f"Expected a replacement sequence of length {len(wt_sequence_str)}, got {len(replacement_sequence)}."
        )

    model = dna_client.create(api_key)
    variant = genome.Variant(
        chromosome="chr20",
        position=44370692,
        reference_bases=wt_sequence_str,
        alternate_bases=replacement_sequence,
        name="generated_enhancer_replacement",
    )

    center = int((44370692 + 44355699) / 2)
    context_interval = genome.Interval("chr20", center, center, strand="+").resize(
        dna_client.SEQUENCE_LENGTH_100KB
    )
    enhancer_interval = genome.Interval("chr20", 44370692, 44370892)
    tss_interval = genome.Interval("chr20", 44355699 - 20, 44355699 + 20, strand="+")

    variant_output = model.predict_variant(
        interval=context_interval,
        variant=variant,
        requested_outputs={
            dna_client.OutputType.ATAC,
            dna_client.OutputType.DNASE,
            dna_client.OutputType.CAGE,
        },
        ontology_terms=["EFO:0001187"],
    )

    def summarize_track(label: str, ref_values, alt_values):
        ref_values = np.asarray(ref_values, dtype=np.float64).reshape(-1)
        alt_values = np.asarray(alt_values, dtype=np.float64).reshape(-1)
        delta = alt_values - ref_values
        return {
            "track": label,
            "n_positions": int(delta.size),
            "ref_total_signal": float(ref_values.sum()),
            "alt_total_signal": float(alt_values.sum()),
            "total_signal_change": float(delta.sum()),
            "mean_signal_change_per_position": float(delta.mean()),
        }

    atac_ref = variant_output.reference.atac.slice_by_interval(enhancer_interval).values.flatten()
    atac_alt = variant_output.alternate.atac.slice_by_interval(enhancer_interval).values.flatten()
    dnase_ref = variant_output.reference.dnase.slice_by_interval(enhancer_interval).values.flatten()
    dnase_alt = variant_output.alternate.dnase.slice_by_interval(enhancer_interval).values.flatten()
    cage_ref = variant_output.reference.cage.filter_to_positive_strand().slice_by_interval(tss_interval).values.flatten()
    cage_alt = variant_output.alternate.cage.filter_to_positive_strand().slice_by_interval(tss_interval).values.flatten()

    summary_df = pd.DataFrame(
        [
            summarize_track("ATAC enhancer window", atac_ref, atac_alt),
            summarize_track("DNASE enhancer window", dnase_ref, dnase_alt),
            summarize_track("CAGE TSS window (+ strand)", cage_ref, cage_alt),
        ]
    )
    return summary_df, variant_output

# Use one arbitrary generated sequence as the replacement example;
replacement_sequence = generated_sequences[0]

ALPHAGENOME_API_KEY = os.environ.get("ALPHAGENOME_API_KEY")
if not ALPHAGENOME_API_KEY:
    raise ValueError("Set ALPHAGENOME_API_KEY in the environment before running this cell.")

signal_change_summary, alphagenome_variant_output = build_signal_change_summary(
    replacement_sequence=replacement_sequence,
    api_key=ALPHAGENOME_API_KEY,
)

display(signal_change_summary)


,track,n_positions,ref_total_signal,alt_total_signal,total_signal_change,mean_signal_change_per_position
0,ATAC enhancer window,200,432.057652,449.934303,17.876652,0.089383
1,DNASE enhancer window,200,350.422852,678.068359,327.645508,1.638228
2,CAGE TSS window (+ strand),40,3797.937500,4050.968750,253.031250,6.325781
